# Machine Learning Tutorial Using `scikit-learn`
### Applied to the TB NanoString data
**W. Evan Johnson** · Machine Learning for Biomedical Data (GSND 5355Q)

This is the Python companion to the `caret` tutorial. It uses the same data and runs the
same analyses (KNN, random forest, SVM, neural network, gradient boosting, and a stacked
ensemble), but the goal is to learn how **scikit-learn** thinks. `caret` hides most of the
machinery behind one function, `train()`. scikit-learn instead gives you small, composable
pieces (estimators, transformers, pipelines, CV splitters, scorers, and search objects) that
you snap together yourself. Once you understand those pieces, every model in the library
works the same way.

**What you'll learn**
1. The estimator API: `fit`, `predict`, `predict_proba`, and learned attributes
2. Stratified train/test splits and reproducibility with `random_state`
3. ROC curves and AUC with `sklearn.metrics`
4. Cross-validation splitters, scoring, and `Pipeline` (and why pipelines prevent leakage)
5. Hyperparameter tuning with `GridSearchCV`
6. Six classifiers on the TB data, then a stacked ensemble
7. Comparing models fairly on identical resamples

## 0. The data: TB NanoString panel

The data come from Kaipilyawar et al. (2022, *Clinical Infectious Diseases*), a study of
whether blood gene expression can separate **active tuberculosis (TB)** from **latent TB
infection (LTBI)**. Expression of a targeted panel of 107 TB-related genes was measured on
the NanoString platform.

* Each row is one patient.
* The first column is TB status (TB vs. LTBI), our binary outcome.
* The other 107 columns are normalized expression values, our features.

With many features relative to patients, overfitting is the central risk, which is why
everything below is organized around honest, cross-validated evaluation.

### Setup
If you are on Colab or a fresh environment, install the packages first:
```
pip install scikit-learn pandas matplotlib seaborn pyreadr
```

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyreadr

from sklearn.model_selection import (train_test_split, RepeatedStratifiedKFold,
                                     cross_validate, GridSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_curve, roc_auc_score, RocCurveDisplay,
                             confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, accuracy_score)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              StackingClassifier)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.exceptions import ConvergenceWarning

SEED = 0
sns.set_theme(style="whitegrid", context="notebook")

### Load the data
`pyreadr` reads the same `.rds` file the R tutorial uses, so both versions start from
identical data. It returns a dictionary of data frames; a single-object `.rds` file is
stored under the key `None`.

In [ ]:
TBnanostring = pyreadr.read_r("TBnanostring.rds")[None]
TBnanostring = TBnanostring.rename(columns={TBnanostring.columns[0]: "TB_Status"})
TBnanostring["TB_Status"] = TBnanostring["TB_Status"].astype(str)

print(TBnanostring.shape)
TBnanostring.head()

### From a data frame to `X` and `y`
Here is the first real difference from R. `caret` takes a formula (`TB_Status ~ .`) and a
data frame. scikit-learn takes a **feature matrix `X`** (samples × features) and a
**target vector `y`**, separately. There are no formulas; you decide exactly which columns
go into `X`.

We also encode the outcome explicitly as 0/1 with **TB = 1**. scikit-learn decides which
class is "positive" from the sorted labels, so being explicit avoids silent surprises in
sensitivity, specificity, and ROC orientation.

In [ ]:
X = TBnanostring.drop(columns="TB_Status")
y = (TBnanostring["TB_Status"] == "TB").astype(int)   # 1 = active TB, 0 = LTBI
class_names = ["LTBI", "TB"]                            # index matches the 0/1 code

print("X:", X.shape, "  y:", y.shape)
y.value_counts().rename(index={0: "LTBI", 1: "TB"})

## 1. How scikit-learn is organized

Everything in scikit-learn is an **estimator**: a Python object with a shared interface.

| Step | Method | What it does |
|---|---|---|
| Configure | `Model(param=...)` | Hyperparameters are set when the object is created |
| Learn | `.fit(X, y)` | Estimates parameters from data; returns the fitted object |
| Predict | `.predict(X)` | Hard class labels |
| Score | `.predict_proba(X)` / `.decision_function(X)` | Continuous scores, needed for ROC/AUC |
| Transform | `.transform(X)` | For preprocessors such as `StandardScaler` |

Two conventions to remember:
* **Hyperparameters** (chosen by you) are arguments to the constructor; `.get_params()` lists them.
* **Learned quantities** (estimated from data) end in an underscore: `coef_`, `classes_`,
  `feature_importances_`. They only exist after `.fit()`.

Because the interface is shared, a logistic regression and a random forest use identical code:

In [ ]:
demo_models = {
    "Logistic regression": LogisticRegression(max_iter=5000),
    "Random forest": RandomForestClassifier(random_state=SEED),
}
for name, est in demo_models.items():
    est.fit(X, y)                                   # same call
    p = est.predict_proba(X.iloc[:3])[:, 1]         # same call
    print(f"{name:20s} classes_={est.classes_}  P(TB) first 3 = {np.round(p, 3)}")

(These were fit on all the data just to show the API; they say nothing about performance.)

**Coming from `caret`**, here is a rough translation table:

| caret | scikit-learn |
|---|---|
| `train(y ~ ., method = "rf")` | `RandomForestClassifier().fit(X, y)` |
| `createDataPartition()` | `train_test_split(..., stratify=y)` |
| `trainControl(method = "repeatedcv")` | `RepeatedStratifiedKFold(...)` |
| `metric = "ROC"`, `twoClassSummary` | `scoring="roc_auc"` |
| `preProcess = c("center","scale")` | `Pipeline([("scale", StandardScaler()), ...])` |
| `tuneLength` / `tuneGrid` | `GridSearchCV(param_grid=...)` (you write the grid) |
| `predict(type = "prob")` | `.predict_proba(X)` |
| `confusionMatrix()` | `confusion_matrix()`, `classification_report()` |
| `varImp()` | `feature_importances_`, `permutation_importance()` |
| `caretList()` + `caretStack()` | `StackingClassifier` |
| `set.seed()` | `random_state=` on each object that uses randomness |

The biggest philosophical difference is in the last row of the tuning line: `caret` picks a
grid for you with `tuneLength`; scikit-learn makes you state the grid. That is more typing,
but you always know exactly what was searched.

## 2. Train/test split (70/30)

`train_test_split` with `stratify=y` keeps the TB/LTBI ratio the same in both sets, which
is what `createDataPartition` does in `caret`.

**Reproducibility.** R uses a global seed (`set.seed`). scikit-learn instead takes a
`random_state` argument on every function or object that involves randomness. Pass an
integer and you get the same result every run, independent of whatever else ran before.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED)

print("Train:", X_train.shape, " Test:", X_test.shape)
pd.DataFrame({"train": y_train.value_counts(), "test": y_test.value_counts()}
             ).rename(index={0: "LTBI", 1: "TB"})

The test set is locked away now. Every modeling decision from here on (preprocessing,
hyperparameters, model choice) will be made using the training data only.

## 3. ROC curves and AUC

A classifier that outputs a **score** (a probability or any continuous value) can be turned
into a yes/no rule by choosing a threshold. The ROC curve traces sensitivity (true positive
rate) against 1 − specificity (false positive rate) as that threshold sweeps from high to low.

The **AUC** summarizes the whole curve: it equals the probability that a randomly chosen TB
patient gets a higher score than a randomly chosen LTBI patient. 0.5 is a coin flip, 1.0 is
perfect separation. Because it depends only on how cases are *ranked*, AUC works with any
monotone score, not just calibrated probabilities. We will use that fact for the SVM.

In scikit-learn:
* `roc_curve(y_true, score)` returns the FPR, TPR, and thresholds
* `roc_auc_score(y_true, score)` returns the AUC
* `RocCurveDisplay` draws it

A sanity check with pure noise should give an AUC near 0.5:

In [ ]:
rng = np.random.default_rng(123)
true_labels = rng.integers(0, 2, size=100)
random_scores = rng.uniform(size=100)

fpr, tpr, thresholds = roc_curve(true_labels, random_scores)
auc_val = roc_auc_score(true_labels, random_scores)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot(fpr, tpr, lw=2, color="#2C7BB6", label=f"Random scores (AUC = {auc_val:.3f})")
ax.plot([0, 1], [0, 1], ls="--", color="gray")
ax.set(xlabel="False positive rate (1 − specificity)",
       ylabel="True positive rate (sensitivity)", title="ROC curve for random guessing")
ax.legend(loc="lower right");

### A reusable evaluation helper
`caret`'s `confusionMatrix()` prints accuracy, sensitivity, and specificity in one go.
scikit-learn keeps these as separate building blocks, so we write a small helper once and
reuse it for every model. Writing it yourself is also a good check that you know what each
metric means.

In [ ]:
test_results = {}   # model name -> dict of test-set metrics and scores
cv_scores = {}      # model name -> per-split CV AUCs (filled in as we go)

def positive_scores(model, X):
    """Continuous score for the TB class: predict_proba if available, else decision_function."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)

def evaluate_on_test(model, name, X=X_test, y=y_test):
    pred = model.predict(X)
    score = positive_scores(model, X)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    res = {"Accuracy": accuracy_score(y, pred),
           "Sensitivity": tp / (tp + fn),
           "Specificity": tn / (tn + fp),
           "AUC": roc_auc_score(y, score)}
    test_results[name] = {**res, "score": score}

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    ConfusionMatrixDisplay.from_predictions(y, pred, display_labels=class_names,
                                            cmap="Blues", colorbar=False, ax=axes[0])
    axes[0].set_title(f"{name}: test confusion matrix"); axes[0].grid(False)
    RocCurveDisplay.from_predictions(y, score, name=name, ax=axes[1], plot_chance_level=True)
    axes[1].set(title=f"{name}: test ROC (AUC = {res['AUC']:.3f})",
                xlabel="False positive rate (1 − specificity)", ylabel="True positive rate (sensitivity)")
    plt.tight_layout(); plt.show()
    return pd.Series(res).round(3)

## 4. Resampling, scoring, and pipelines

In `caret`, `trainControl()` bundles the resampling plan, the metric, and what to save.
scikit-learn splits these into separate objects, each with one job.

**The CV splitter** only decides which rows go in which fold. `RepeatedStratifiedKFold`
with 5 folds × 3 repeats gives 15 train/validation splits, each stratified by class. Because
it has a fixed `random_state`, every model below is evaluated on *exactly the same* 15
splits, which makes the later model comparison a paired one.

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
SCORING = "roc_auc"
print(f"{cv.get_n_splits()} train/validation splits")

**Cross-validating a single model.** `cross_validate` fits a fresh copy of the estimator on
each training fold and scores it on the held-out fold. Here is a random forest with default
settings, the analog of the first `train(..., method = "rf")` call in the R tutorial.

In [ ]:
rf_default = RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1)
cv_rf = cross_validate(rf_default, X_train, y_train, cv=cv,
                       scoring=["roc_auc", "accuracy"], return_train_score=True)
cv_scores["RF (default)"] = cv_rf["test_roc_auc"]

pd.DataFrame(cv_rf)[["train_roc_auc", "test_roc_auc", "test_accuracy"]].agg(["mean", "std"]).round(3)

Notice `train_roc_auc` is essentially 1.0 while the held-out AUC is lower. A random forest
can memorize its training data, so training performance is useless for judging it. This
gap is exactly why we resample.

Cross-validation *estimates* performance; it does not produce the final model. To get a
model we can use, we fit once on the full training set and then check it on the test set.
(`caret` does this refit automatically; in scikit-learn, `cross_validate` doesn't, but
`GridSearchCV` below does.)

In [ ]:
rf_default.fit(X_train, y_train)
evaluate_on_test(rf_default, "RF (default)")

### Pipelines: preprocessing that respects cross-validation

KNN, SVMs, and neural networks are sensitive to feature scale, so we standardize genes to
mean 0, SD 1. The trap is doing this *before* splitting into folds: the scaler would then
learn means and SDs from the validation rows, leaking information and inflating CV scores.

A `Pipeline` chains steps into one estimator. When CV calls `pipe.fit(X_fold_train)`, the
scaler is fit on that fold's training rows only, then applied to its validation rows. This
is what `caret`'s `preProcess` argument does behind the scenes; in scikit-learn, you build
it yourself, and it works the same way for any preprocessing step (imputation, filtering,
PCA, batch correction...).

Each step has a name, and hyperparameters are addressed as `stepname__parameter`
(two underscores). We'll use that syntax when tuning.

In [ ]:
example_pipe = Pipeline([("scale", StandardScaler()),
                         ("knn", KNeighborsClassifier())])
example_pipe

### Hyperparameter tuning with `GridSearchCV`
`GridSearchCV` wraps an estimator (or pipeline) together with a parameter grid and a CV
splitter. Calling `.fit()`:
1. cross-validates every combination in the grid on the training data,
2. picks the combination with the best mean CV score (`best_params_`, `best_score_`),
3. refits that combination on the full training set (`best_estimator_`).

The fitted search object behaves like the best model, so `.predict()` and
`.predict_proba()` just work. Every fold's score is kept in `cv_results_`.

A small helper to turn `cv_results_` into a tidy table:

In [ ]:
def tuning_table(search):
    res = pd.DataFrame(search.cv_results_)
    keep = [c for c in res.columns if c.startswith("param_")] + ["mean_test_score", "std_test_score"]
    out = res[keep].rename(columns=lambda c: c.replace("param_", ""))
    return out.sort_values("mean_test_score", ascending=False)

def store_cv(search, name):
    idx = search.best_index_
    cv_scores[name] = np.array([search.cv_results_[f"split{i}_test_score"][idx]
                                for i in range(cv.get_n_splits())])
    print(f"Best params: {search.best_params_}\nMean CV AUC: {search.best_score_:.3f}")

## 5. k-Nearest Neighbors (KNN)

KNN labels a patient by majority vote among the *k* most similar training patients.
"Similar" means Euclidean distance across all 107 genes, so a gene with a large numeric range
would dominate unless we standardize. Hence the pipeline.

The key hyperparameter is *k*: small *k* gives a flexible, noisy boundary; large *k* gives a
smooth one. We also try `weights="distance"`, which lets closer neighbors count more.

In [ ]:
knn_pipe = Pipeline([("scale", StandardScaler()),
                     ("knn", KNeighborsClassifier())])
knn_grid = {"knn__n_neighbors": list(range(3, 32, 2)),
            "knn__weights": ["uniform", "distance"]}

knn_search = GridSearchCV(knn_pipe, knn_grid, cv=cv, scoring=SCORING, n_jobs=-1)
knn_search.fit(X_train, y_train)
store_cv(knn_search, "KNN")

In [ ]:
knn_tab = tuning_table(knn_search)
fig, ax = plt.subplots(figsize=(6, 4))
for w, d in knn_tab.groupby("knn__weights"):
    d = d.sort_values("knn__n_neighbors")
    ax.errorbar(d["knn__n_neighbors"], d["mean_test_score"], yerr=d["std_test_score"],
                marker="o", capsize=3, label=f"weights = {w}")
ax.set(xlabel="k (number of neighbors)", ylabel="CV AUC (mean ± SD over 15 splits)",
       title="KNN tuning profile"); ax.legend();

In [ ]:
evaluate_on_test(knn_search, "KNN")

Error bars show how much the AUC varies from split to split. Settings whose bars overlap
heavily are not really distinguishable, so don't over-read the exact best *k*.

## 6. Random forest

A random forest grows many decision trees on bootstrap samples and, at each split, only
considers a random subset of features. Averaging many de-correlated trees reduces variance.
Trees split on thresholds, so scaling is unnecessary.

The main tuning parameter is `max_features`, the number of genes tried at each split
(`mtry` in R's `randomForest`). The default for classification is √p ≈ 10.

**Out-of-bag (OOB) error.** Each tree never sees about a third of the training rows, so
those rows give a free internal validation estimate. Set `oob_score=True` to get it.

In [ ]:
p = X_train.shape[1]
rf_grid = {"max_features": [2, int(np.sqrt(p)), 25, 54, p]}

rf_search = GridSearchCV(
    RandomForestClassifier(n_estimators=500, oob_score=True, random_state=SEED, n_jobs=-1),
    rf_grid, cv=cv, scoring=SCORING, n_jobs=-1)
rf_search.fit(X_train, y_train)
store_cv(rf_search, "Random forest")
print(f"OOB accuracy of refit model: {rf_search.best_estimator_.oob_score_:.3f}")
tuning_table(rf_search).round(3)

In [ ]:
evaluate_on_test(rf_search, "Random forest")

### Variable importance: two different questions
scikit-learn offers two importance measures, and they answer different questions:

* **`feature_importances_` (impurity-based)**: how much each gene reduced node impurity,
  summed over all trees. Fast and computed on training data, but biased toward features
  with many possible split points, and it can credit genes the model merely memorized.
* **`permutation_importance`**: shuffle one gene and measure how much the AUC drops on
  data the model has not seen. Slower, but it measures what we care about: reliance on
  the gene for *generalization*. Correlated genes can share (and dilute) credit.

Comparing both is good practice.

In [ ]:
rf_best = rf_search.best_estimator_
imp = pd.DataFrame({"Impurity": rf_best.feature_importances_}, index=X.columns)

perm = permutation_importance(rf_best, X_test, y_test, scoring="roc_auc",
                              n_repeats=30, random_state=SEED, n_jobs=-1)
imp["Permutation"] = perm.importances_mean

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, col in zip(axes, imp.columns):
    top = imp[col].sort_values().tail(10)
    ax.barh(top.index, top.values, color="#2C7BB6")
    ax.set(title=f"Top 10 genes: {col.lower()} importance", xlabel=col + " importance")
plt.tight_layout();

## 7. Support vector machine (SVM)

An SVM finds the boundary that separates the classes with the widest possible margin. The
**RBF (radial) kernel** lets that boundary curve by implicitly mapping samples into a richer
feature space. Two hyperparameters interact:

* `C`: the penalty for margin violations. Large `C` fits the training data tightly (risking
  overfitting); small `C` tolerates more errors for a smoother boundary.
* `gamma`: how far one training point's influence reaches. Large `gamma` means very local,
  wiggly boundaries. (This is `sigma` in `kernlab`/`caret`.) `"scale"` uses
  1 / (p · var(X)), a sensible default.

Because both interact, we search a 2-D grid and look at it as a heatmap. SVMs are
distance-based, so the scaler goes in the pipeline.

**Scores for ROC.** `SVC` produces a signed distance from the boundary via
`decision_function`. That is a perfectly good score for AUC (only the ranking matters). We
skip `probability=True`, which would fit an extra internal 5-fold calibration model on every
fit; for this purpose it is slower and adds nothing.

In [ ]:
svm_pipe = Pipeline([("scale", StandardScaler()),
                     ("svm", SVC(kernel="rbf"))])
svm_grid = {"svm__C": [0.25, 0.5, 1, 2, 4, 8, 16, 32, 64, 128],
            "svm__gamma": ["scale", 1e-4, 1e-3, 1e-2]}

svm_search = GridSearchCV(svm_pipe, svm_grid, cv=cv, scoring=SCORING, n_jobs=-1)
svm_search.fit(X_train, y_train)
store_cv(svm_search, "SVM (RBF)")

In [ ]:
heat = (tuning_table(svm_search)
        .assign(svm__gamma=lambda d: d["svm__gamma"].astype(str))
        .pivot(index="svm__gamma", columns="svm__C", values="mean_test_score"))
fig, ax = plt.subplots(figsize=(8, 3))
sns.heatmap(heat, annot=True, fmt=".3f", cmap="viridis", ax=ax, cbar_kws={"label": "CV AUC"})
ax.set(xlabel="C", ylabel="gamma", title="SVM tuning surface");

In [ ]:
evaluate_on_test(svm_search, "SVM (RBF)")

## 8. Neural network (multilayer perceptron)

`MLPClassifier` is a feed-forward neural network. With one hidden layer it is the same model
as R's `nnet`: inputs → a few hidden units with nonlinear activations → output probability.

| `nnet` (caret) | `MLPClassifier` |
|---|---|
| `size` (hidden units) | `hidden_layer_sizes=(size,)` |
| `decay` (weight decay) | `alpha` (L2 penalty) |
| BFGS optimizer | `solver="lbfgs"`, the best choice for small datasets |

With 107 inputs and only ~100 training patients, even a modest network has more weights than
samples, so the `alpha` penalty does a lot of the work. Neural nets are sensitive to input
scale, so we standardize.

scikit-learn's MLP is meant for small tabular problems like this one. For images, sequences,
or large data you'd move to PyTorch or similar.

In [ ]:
mlp_pipe = Pipeline([("scale", StandardScaler()),
                     ("mlp", MLPClassifier(solver="lbfgs", max_iter=2000, random_state=SEED))])
mlp_grid = {"mlp__hidden_layer_sizes": [(1,), (3,), (5,), (7,), (9,)],
            "mlp__alpha": [1e-4, 1e-2, 1e-1, 1.0, 10.0]}

mlp_search = GridSearchCV(mlp_pipe, mlp_grid, cv=cv, scoring=SCORING, n_jobs=-1)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    mlp_search.fit(X_train, y_train)
store_cv(mlp_search, "Neural net")

In [ ]:
mlp_tab = tuning_table(mlp_search).assign(
    units=lambda d: d["mlp__hidden_layer_sizes"].map(lambda t: t[0]))
fig, ax = plt.subplots(figsize=(6, 4))
for a, d in mlp_tab.groupby("mlp__alpha"):
    d = d.sort_values("units")
    ax.plot(d["units"], d["mean_test_score"], marker="o", label=f"alpha = {a:g}")
ax.set(xlabel="Hidden units", ylabel="CV AUC", title="MLP tuning profile"); ax.legend(fontsize=8);

In [ ]:
evaluate_on_test(mlp_search, "Neural net")

Look at how strongly the curves depend on `alpha`: regularization matters more than network
size here. That's typical when features outnumber samples.

## 9. Gradient boosting

Boosting builds shallow trees *sequentially*; each new tree is fit to the errors of the
ensemble so far, and its contribution is shrunk by a learning rate. Where a random forest
reduces variance by averaging independent trees, boosting reduces bias by correcting
mistakes step by step.

| `gbm` (caret) | `GradientBoostingClassifier` |
|---|---|
| `n.trees` | `n_estimators` |
| `interaction.depth` | `max_depth` |
| `shrinkage` | `learning_rate` |
| `n.minobsinnode` | `min_samples_leaf` |

The number of trees and the learning rate trade off: a smaller learning rate needs more
trees. `subsample < 1` (stochastic boosting) adds randomness that usually helps.

For larger datasets, `HistGradientBoostingClassifier` is much faster and supports built-in
early stopping. We use the classic version here because it mirrors `gbm`.

In [ ]:
gbm_grid = {"n_estimators": [100, 250, 500],
            "max_depth": [1, 2, 3],
            "learning_rate": [0.01, 0.1]}

gbm_search = GridSearchCV(
    GradientBoostingClassifier(subsample=0.8, min_samples_leaf=5, random_state=SEED),
    gbm_grid, cv=cv, scoring=SCORING, n_jobs=-1)
gbm_search.fit(X_train, y_train)
store_cv(gbm_search, "Gradient boosting")

In [ ]:
gbm_tab = tuning_table(gbm_search)
g = sns.relplot(data=gbm_tab, x="n_estimators", y="mean_test_score", hue="max_depth",
                col="learning_rate", kind="line", marker="o", palette="viridis",
                height=3.5, aspect=1.1)
g.set_axis_labels("Number of trees", "CV AUC").figure.suptitle("Boosting tuning profile", y=1.03);

In [ ]:
evaluate_on_test(gbm_search, "Gradient boosting")

In [ ]:
gbm_imp = pd.Series(gbm_search.best_estimator_.feature_importances_, index=X.columns)
top20 = gbm_imp.sort_values().tail(20)
fig, ax = plt.subplots(figsize=(5, 6))
ax.barh(top20.index, top20.values, color="#2C7BB6")
ax.set(title="Top 20 genes: boosting importance", xlabel="Impurity importance");

## 10. Stacked ensemble

Different models make different mistakes. **Stacking** trains a *meta-learner* that takes
the base models' predicted scores as its inputs and learns how best to combine them.

The subtle part is that the meta-learner must be trained on predictions the base models
made for data they *didn't* train on; otherwise it learns to trust whichever model
overfits most. `StackingClassifier` handles this with its own internal CV: it generates
out-of-fold predictions for every training row (this is what `caretList` saves with
`savePredictions`), fits the meta-learner on those, then refits each base model on all the
training data.

As in the R tutorial, we stack KNN, the SVM, and boosting, with a random forest on top.
We pass the *tuned* configurations (`best_estimator_`) rather than the search objects, so
the stack doesn't redo every grid search inside every fold.

In [ ]:
base_models = [("knn", knn_search.best_estimator_),
               ("svm", svm_search.best_estimator_),
               ("gbm", gbm_search.best_estimator_)]

stack = StackingClassifier(
    estimators=base_models,
    final_estimator=RandomForestClassifier(n_estimators=500, random_state=SEED),
    cv=5, stack_method="auto", n_jobs=-1)

# CV estimate of the whole stacking procedure, on the same 15 splits as everything else
stack_cv = cross_validate(stack, X_train, y_train, cv=cv, scoring=SCORING)
cv_scores["Stacked ensemble"] = stack_cv["test_score"]
print(f"Mean CV AUC (stack): {stack_cv['test_score'].mean():.3f}")

stack.fit(X_train, y_train)
evaluate_on_test(stack, "Stacked ensemble")

`stack_method="auto"` uses `predict_proba` when a base model has it and falls back to
`decision_function` otherwise (our SVM). The meta-learner never needs those to be on the
same scale; a tree-based meta-learner only uses their ordering.

One honest caveat: the base models' hyperparameters were tuned on the full training set,
so this CV estimate of the stack is slightly optimistic. Fully nested CV would fix that at
a large computational cost.

## 11. Comparing the models

Every model was cross-validated on the same 15 splits, so their CV AUCs can be compared
split by split, which removes the noise from "which patients happened to land in which
fold". This is the scikit-learn analog of `caret::resamples()`.

In [ ]:
cv_df = pd.DataFrame(cv_scores)
order = cv_df.mean().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=cv_df[order], orient="h", color="#BFD7EA", ax=ax, showfliers=False)
sns.stripplot(data=cv_df[order], orient="h", color="#2C7BB6", size=4, ax=ax)
ax.set(xlabel="CV AUC (15 resampled splits)", title="Cross-validated performance by model");

In [ ]:
summary = pd.DataFrame({k: {m: v for m, v in d.items() if m != "score"}
                        for k, d in test_results.items()}).T
summary.insert(0, "CV AUC (mean)", cv_df.mean())
summary.round(3).sort_values("CV AUC (mean)", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
for name in order:
    RocCurveDisplay.from_predictions(y_test, test_results[name]["score"],
                                     name=name, ax=ax)
ax.plot([0, 1], [0, 1], ls="--", color="gray")
ax.set(title="Test-set ROC curves", xlabel="False positive rate (1 − specificity)",
       ylabel="True positive rate (sensitivity)"); ax.legend(fontsize=8, loc="lower right");

**Reading these results.** The test set has only about 50 patients, so test AUCs carry
substantial uncertainty; differences of a few hundredths between models are usually noise.
The CV distributions are the better guide to choosing a model, and the test set is the
final, one-time check that the chosen model generalizes. If you find yourself going back to
tweak models after looking at test results, the test set is no longer an independent check.

## 12. Environment

In [ ]:
import sklearn, sys
print("Python", sys.version.split()[0])
for mod in (sklearn, np, pd, sns, pyreadr):
    print(f"{mod.__name__:10s} {mod.__version__}")